# Block 1 — live lecture demo

Instructor notebook for the 15-minute Block 1 lecture. Run top to bottom.

This is the same material as `lecture_notes.md` §§1–4, paced for narration.
Your exercise is `read_explore_exercise.ipynb`.

## 1 · Load — three lines

In [ ]:
import odmlib.define_loader as DL
import odmlib.loader as LD

loader = LD.ODMLoader(DL.XMLDefineLoader(model_package="define_2_1"))
loader.open_odm_document("../data/defineV21-SDTM.xml")

odm = loader.root()                 # call root() ONCE and keep the reference
mdv = odm.Study.MetaDataVersion     # single objects in Define-XML - no [0]

`model_package="define_2_1"` is required — v2.0 is the default.

`Study` and `MetaDataVersion` are single objects: one study, one metadata version
per define.xml. (In ODM 1.3.2 they are lists.)

In [ ]:
print("Study:      ", odm.Study.GlobalVariables.StudyName)
print("MetaDataVer:", mdv.Name)
print("Define ver: ", mdv.DefineVersion)

There is also a context-manager shortcut that defaults to Define-XML v2.1:

In [ ]:
from odmlib.context import open_define

with open_define("../data/defineV21-SDTM.xml") as define:
    print(define.Study.MetaDataVersion.OID)

## 2 · Navigate — the collections are plain Python lists

In [ ]:
for igd in mdv.ItemGroupDef:
    print(f"{igd.OID:<12} {igd.Name:<8} {len(igd.ItemRef):>3} variables   {igd.Structure}")

`def:Structure` is just `.Structure` — namespaces disappear at the Python level.

In [ ]:
metrics = {
    "datasets": len(mdv.ItemGroupDef),
    "variables": len(mdv.ItemDef),
    "codelists": len(mdv.CodeList),
    "methods": len(mdv.MethodDef),
}
for name, count in metrics.items():
    print(f"{name:>10}: {count}")

## 3 · Find — one call instead of a nested loop

`find(class_name, attribute, value)` searches all descendants, returns the first
match or `None`. `find_all()` returns every match.

In [ ]:
age = mdv.find("ItemDef", "OID", "IT.DM.AGE")

print("Name:       ", age.Name)
print("DataType:   ", age.DataType)
print("Length:     ", age.Length)
print("Description:", age.Description.TranslatedText[0]._content)   # _content, not .text
print("Origin:     ", age.Origin[0].Type)

## 4 · Follow a reference — `ItemRef` → `ItemDef`

The graph edge from the mental-model slide. `ItemRef` holds the dataset-specific
facts (order, mandatory, key); the `ItemDef` holds the variable-level facts.

In [ ]:
vs = mdv.find("ItemGroupDef", "OID", "IG.VS")

print(f"{'#':>3} {'Name':<10} {'Type':<10} {'Len':>4} {'Mandatory'}")
for ref in sorted(vs.ItemRef, key=lambda r: int(r.OrderNumber)):
    item = mdv.find("ItemDef", "OID", ref.ItemOID)
    length = item.Length if item.Length is not None else ""
    print(f"{ref.OrderNumber:>3} {item.Name:<10} {item.DataType:<10} {length!s:>4} {ref.Mandatory}")

Two hops for a codelist: `ItemDef` → `CodeListRef.CodeListOID` → `CodeList`.

In [ ]:
vstestcd = mdv.find("ItemDef", "OID", "IT.VS.VSTESTCD")
cl = mdv.find("CodeList", "OID", vstestcd.CodeListRef.CodeListOID)

print(f"{vstestcd.Name} uses {cl.OID} ({cl.Name}):")
for term in cl.CodeListItem[:5]:
    print(f"   {term.CodedValue:<10} = {term.Decode.TranslatedText[0]._content}")
for term in cl.EnumeratedItem[:5]:      # enumerations have no decodes - check both
    print(f"   {term.CodedValue}")

### Reverse lookup — who carries this OID?

Build the OID index once, then query it.

In [ ]:
idx = odm.build_oid_index()
for obj in idx.find_all("IT.DM.AGE"):
    print(type(obj).__name__)

---

**Your turn:** `read_explore_exercise.ipynb` — 4 TODOs, ~20 minutes.
Keep `lecture_notes.md` open beside it.